In [1]:
import pandas as pd
import sys
import os
sys.path.insert(0, '/home/hugo/gitlab.pavlovia/online_test/code')
from my_analysis.data_loader import get_clip_duration
from my_analysis.utils import calculate_dice
from sklearn.metrics import cohen_kappa_score
import numpy as np
import datetime
import plotly.express as px
import glob
import re
import json


# sub-01 from reference panel

In [ ]:
def load_and_aggregate_reference(path_data,id=str, cutoff=datetime.datetime(2026, 3, 25), save_data=False):
    """
    Loads all CSV files in the specified directory and aggregates them into a single DataFrame.
    
    Args:
        path_data (str): Path to the directory containing the data CSV files.
        
    Returns:
        pd.DataFrame: A long-format DataFrame containing the aggregated data.
    """
    all_files = glob.glob(os.path.join(path_data, "*.csv"))
    files_sorted = sorted(all_files, key=os.path.getmtime)[::-1]
    print(len(files_sorted), 'files were collected')
    aggregated_data = []
    scene_players = []
    for filename in files_sorted:
            try:
                df = pd.read_csv(filename)
            except Exception as e:
                 print(e)
            try:
                    # Extract Judge ID
                    judge_id = id
                    participant = id
                    
                    # Extract Scene ID
                    raw_scene_id = re.search(r'w(\d+)l(\d+)_scene-(\d+)', df['clip_path'].dropna().iloc[0]).group(0)
                    scene_id = str(raw_scene_id).replace("_scene-", "s")
                    scene_id = scene_id.strip("'").strip('"').strip("’").strip("‘")
                    
                    # Check completed status
                    done = False
                    if 'thank_s.started' in df.columns and 'thank_s.stopped' in df.columns:
                        if not df['thank_s.started'].dropna().empty and not df['thank_s.stopped'].dropna().empty:
                            done = True
                    
                    # Process answers
                    answers_col = 'anwsers' if 'anwsers' in df.columns else 'answers'
                    
                    if answers_col in df.columns and not df[answers_col].dropna().empty:
                        answers_json = df[answers_col].dropna().iloc[-1]
                        try:
                            answers_dict = json.loads(answers_json)
                            players = list(answers_dict.keys())
                            id_1_s_p = scene_id+ '_'+players[0]+'_'+players[1]
                            id_2_s_p = scene_id+ '_'+players[1]+'_'+players[0]
                            if id_1_s_p in scene_players or id_2_s_p in scene_players:
                                 raise ValueError(id_1_s_p, "already present")
                            else:
                                 scene_players.append(id_1_s_p)
                                 scene_players.append(id_2_s_p)

                            # Structure: player -> question -> clip -> answer (bool)
                            for player, questions in answers_dict.items():
                                for question, clips in questions.items():
                                    for clip_key, answer in clips.items():
                                        try:
                                            clip_num = int(clip_key.split('_')[1])-1
                                        except (IndexError, ValueError):
                                            continue 

                                        if player=='ppo':
                                            path_pattern = 'ppo_mario_ep-'
                                        elif 'im' in player:
                                            path_pattern = player[4:]+'_epoch='
                                        else:
                                            path_pattern = player
                                        
                                        clip_row = df[
                                            (df['clip_num'] == clip_num) &
                                            (df['clip_path'].str.split('/').str[1].str.contains(path_pattern))
                                            ]
                                        if not clip_row.empty:
                                            vis_times = []
                                            clip_path_rel = clip_row['clip_path'].iloc[0]
                                            json_path_rel = clip_row['clip_path'].iloc[0].replace('videos', 'infos').replace('.mp4', '.json')
                                            
                                            for idx in clip_row.index:
                                                v_start = df.at[idx, 'display_clips.started']
                                                v_stop = df.at[idx, 'display_clips.stopped']
                                                
                                                if pd.notnull(v_start) and pd.notnull(v_stop):
                                                    vis_times.append(v_stop - v_start)
                                                
                                            vis_time = max(vis_times) if vis_times else None

                                            with open(json_path_rel) as json_data:
                                                d = json.load(json_data)
                                                cleared = d['Cleared']

                                            video_duration = get_clip_duration(clip_path_rel, cleared)
                                            if answer==False or answer==True:
                                                 pass
                                            else:
                                                 print(answer, 'dans le fichier : ',filename, ' question : ',question, ' joueur : ',player, ' scene : ', scene_id, ' question : ',question, ' joueur : ',player)

                                            aggregated_data.append({
                                                'participant': participant,
                                                'judge_ID': judge_id,
                                                'scene_id': scene_id,
                                                'player': player,
                                                'question': question,
                                                'clip': clip_num,
                                                'answer': answer,
                                                'visualisation_time': vis_time,
                                                'clip_path': clip_path_rel,
                                                'video_duration': video_duration,
                                                'done': done,
                                            })
                                            
                        except json.JSONDecodeError:
                            print(f"Error decoding JSON for file {filename}")
                            continue

            except Exception as e:
                    print(f"Error processing file {filename}: {e}")
                    continue
    data = pd.DataFrame(aggregated_data)

    return data

In [3]:
data_path = os.path.join('/home/hugo/gitlab.pavlovia/online_test', 'sub-01_data')
#output_dir = os.path.join('/home/hugo/gitlab.pavlovia/online_test', 'outputdata')

df_sub01 = load_and_aggregate_reference(data_path, id='sub-01', cutoff=datetime.datetime(2026, 4, 14))
df_sub01.head()

29 files were collected
Error processing file /home/hugo/gitlab.pavlovia/online_test/sub-01_data/098787_online_test_2026-04-20_14h15.23.983.csv: 'clip_path'
Error processing file /home/hugo/gitlab.pavlovia/online_test/sub-01_data/121089_online_test_2026-04-19_23h10.31.093.csv: ('w5l2s11_im-sub-02_sub-02', 'already present')
Error processing file /home/hugo/gitlab.pavlovia/online_test/sub-01_data/644132_online_test_2026-04-19_22h13.03.175.csv: ('w5l2s11_sub-02_im-sub-02', 'already present')
Error processing file /home/hugo/gitlab.pavlovia/online_test/sub-01_data/052588_online_test_2026-04-19_21h39.18.084.csv: ('w2l3s2_sub-06_ppo', 'already present')
Error processing file /home/hugo/gitlab.pavlovia/online_test/sub-01_data/305464_online_test_2026-04-19_21h27.15.436.csv: ('w2l3s2_sub-06_ppo', 'already present')
Error processing file /home/hugo/gitlab.pavlovia/online_test/sub-01_data/728114_online_test_2026-04-17_19h09.59.699.csv: ('w5l2s2_sub-05_im-sub-05', 'already present')
Error process

,participant,judge_ID,scene_id,player,question,clip,answer,visualisation_time,clip_path,video_duration,done
0,sub-01,sub-01,w5l2s10,im-sub-02,Q_exH,0,True,21.433762,sourcedata/sub-02_epoch=0-step=500/sub-02/ses-...,1.516667,True
1,sub-01,sub-01,w5l2s10,im-sub-02,Q_exH,1,False,9.316853,sourcedata/sub-02_epoch=0-step=500/sub-02/ses-...,4.716667,True
2,sub-01,sub-01,w5l2s10,im-sub-02,Q_exH,2,True,7.333480,sourcedata/sub-02_epoch=0-step=500/sub-02/ses-...,0.583333,True
3,sub-01,sub-01,w5l2s10,im-sub-02,Q_exH,3,True,6.350127,sourcedata/sub-02_epoch=0-step=500/sub-02/ses-...,1.983333,True
4,sub-01,sub-01,w5l2s10,im-sub-02,Q_exH,4,True,6.050121,sourcedata/sub-02_epoch=0-step=500/sub-02/ses-...,2.133333,True


In [5]:
df_analysis_01 = df_sub01[df_sub01['done']].copy()
df_analysis_01['clip_uid'] = df_analysis_01['scene_id'].astype(str) + "_" + df_analysis_01['player'].astype(str) + "_" + df_analysis_01['clip'].astype(str)
print(df_analysis_01['scene_id'].nunique())
for scene in df_analysis_01['scene_id'].unique():
    df_f = df_analysis_01[df_analysis_01['scene_id']==scene]
    print('For scene ',scene,' they are', df_f['participant'].nunique(),'judge')
df_analysis_01['scene_id'].value_counts()

16
For scene  w5l2s10  they are 1 judge
For scene  w2l3s12  they are 1 judge
For scene  w5l2s11  they are 1 judge
For scene  w2l3s4  they are 1 judge
For scene  w2l3s2  they are 1 judge
For scene  w4l1s7  they are 1 judge
For scene  w3l1s1  they are 1 judge
For scene  w8l3s1  they are 1 judge
For scene  w2l3s11  they are 1 judge
For scene  w4l1s6  they are 1 judge
For scene  w7l3s2  they are 1 judge
For scene  w5l2s2  they are 1 judge
For scene  w2l3s8  they are 1 judge
For scene  w7l3s4  they are 1 judge
For scene  w3l2s3  they are 1 judge
For scene  w3l2s1  they are 1 judge


scene_id
w7l3s2     800
w2l3s2     696
w4l1s7     592
w3l2s3     584
w5l2s10    400
w3l1s1     400
w8l3s1     400
w7l3s4     400
w2l3s11    384
w5l2s2     384
w2l3s4     352
w4l1s6     320
w2l3s8     312
w5l2s11    296
w3l2s1     264
w2l3s12    240
Name: count, dtype: int64

In [6]:
df_analysis_01[df_analysis_01.duplicated(keep=False)]

,participant,judge_ID,scene_id,player,question,clip,answer,visualisation_time,clip_path,video_duration,done,clip_uid


# load data sub-02

In [7]:
data_path = os.path.join('/home/hugo/gitlab.pavlovia/online_test', 'sub-02_data')
#output_dir = os.path.join('/home/hugo/gitlab.pavlovia/online_test', 'outputdata')

df = load_and_aggregate_reference(data_path, id='sub-02')
df.head()


21 files were collected
No columns to parse from file
Error processing file /home/hugo/gitlab.pavlovia/online_test/sub-02_data/434434_online_test_2026-04-20_14h11.10.695.csv: ('w3l2s3_sub-06_im-sub-06', 'already present')


,participant,judge_ID,scene_id,player,question,clip,answer,visualisation_time,clip_path,video_duration,done
0,sub-02,sub-02,w4l1s7,sub-03,Q_exH,0,True,15.3108,sourcedata/sub-03/ses-003/beh/videos/sub-03_se...,5.566667,True
1,sub-02,sub-02,w4l1s7,sub-03,Q_exH,1,True,7.1960,sourcedata/sub-03/ses-003/beh/videos/sub-03_se...,6.683333,True
2,sub-02,sub-02,w4l1s7,sub-03,Q_exH,2,False,7.6948,sourcedata/sub-03/ses-003/beh/videos/sub-03_se...,5.883333,True
3,sub-02,sub-02,w4l1s7,sub-03,Q_exH,3,False,8.8797,sourcedata/sub-03/ses-003/beh/videos/sub-03_se...,5.533333,True
4,sub-02,sub-02,w4l1s7,sub-03,Q_exH,4,False,5.4798,sourcedata/sub-03/ses-003/beh/videos/sub-03_se...,5.550000,True


In [8]:
df_analysis_02 = df[df['done']].copy()
df_analysis_02['clip_uid'] = df_analysis_02['scene_id'].astype(str) + "_" + df_analysis_02['player'].astype(str) + "_" + df_analysis_02['clip'].astype(str)
print(df_analysis_02['scene_id'].nunique())
for scene in df_analysis_02['scene_id'].unique():
    df_f = df_analysis_02[df_analysis_02['scene_id']==scene]
    print('For scene ',scene,' they are', df_f['participant'].nunique(),'judge')
df_analysis_02['scene_id'].value_counts()

16
For scene  w4l1s7  they are 1 judge
For scene  w3l2s3  they are 1 judge
For scene  w7l3s2  they are 1 judge
For scene  w5l2s11  they are 1 judge
For scene  w2l3s2  they are 1 judge
For scene  w2l3s4  they are 1 judge
For scene  w3l2s1  they are 1 judge
For scene  w3l1s1  they are 1 judge
For scene  w7l3s4  they are 1 judge
For scene  w5l2s2  they are 1 judge
For scene  w2l3s8  they are 1 judge
For scene  w4l1s6  they are 1 judge
For scene  w2l3s11  they are 1 judge
For scene  w2l3s12  they are 1 judge
For scene  w8l3s1  they are 1 judge
For scene  w5l2s10  they are 1 judge


scene_id
w7l3s2     800
w2l3s2     696
w4l1s7     592
w3l2s3     584
w3l1s1     400
w7l3s4     400
w8l3s1     400
w5l2s10    400
w5l2s2     384
w2l3s11    384
w2l3s4     352
w4l1s6     320
w2l3s8     312
w5l2s11    296
w3l2s1     264
w2l3s12    240
Name: count, dtype: int64

In [9]:
df_analysis_02[df_analysis_02.duplicated(keep=False)]

,participant,judge_ID,scene_id,player,question,clip,answer,visualisation_time,clip_path,video_duration,done,clip_uid


In [10]:
df_analysis = pd.concat([df_analysis_01, df_analysis_02])
judge_metrics = {j: {'percent': [], 'kappa': [], 'dice': []} for j in df_analysis['judge_ID'].unique()}

for (scene, player), group in df_analysis.groupby(['scene_id', 'player']):
        # Pivot answers: Index=[question, clip], Columns=judge_ID
        pivoted = group.pivot_table(index=['question', 'clip'], columns='judge_ID', values='answer')
        for judge in pivoted.columns:
            others = pivoted.drop(columns=[judge])
            consensus = others.mode(axis=1).iloc[:, 0]
            judge_ans = pivoted[judge]
            valid_mask = judge_ans.notna() & consensus.notna()

            if not valid_mask.any(): continue
            
            v1 = judge_ans[valid_mask].astype(int)
            v2 = consensus[valid_mask].astype(int)
            
            judge_metrics[judge]['percent'].append((v1 == v2).mean())
            try:
                k = cohen_kappa_score(v1, v2)
                if np.isnan(k): k = 1.0 
                judge_metrics[judge]['kappa'].append(k)
            except: pass
            judge_metrics[judge]['dice'].append(calculate_dice(v1, v2))
    # Average metrics per judge
final_judge_metrics = []
for judge, measurements in judge_metrics.items():
        row = {'judge_ID': judge}
        for m in ['percent', 'kappa', 'dice']:
            row[m] = np.mean(measurements[m]) if measurements[m] else np.nan
        final_judge_metrics.append(row)           
df_metrics = pd.DataFrame(final_judge_metrics)
df_metrics

,judge_ID,percent,kappa,dice
0,sub-01,0.896002,0.781049,0.865653
1,sub-02,0.896002,0.781049,0.865653


In [20]:

df_concensus = df_analysis[['judge_ID', 'scene_id',	'player', 'question', 'clip', 'answer']]
df_concensus['answer'] = df_concensus['answer'].astype(float)
df_concensus.loc[df_concensus['judge_ID'] == 'sub-01', 'answer'] *= 0.1

In [21]:
print(df_concensus['answer'].sum())
df_concensus_gb = df_concensus.groupby(['scene_id',	'player', 'question', 'clip'], as_index=False).agg({
    'answer':'sum'
})
df_concensus_gb[df_concensus_gb['answer']==2]

2895.8


,scene_id,player,question,clip,answer
3763,w4l1s7,sub-03,Q_exL,13,2.0
3764,w4l1s7,sub-03,Q_exL,14,2.0
3775,w4l1s7,sub-03,Q_exL,25,2.0
3777,w4l1s7,sub-03,Q_exL,27,2.0
5415,w7l3s2,sub-05,Q_exL,11,2.0


In [18]:

px.histogram(df_concensus_gb, x='answer')

In [27]:
df_concensus_gb['answer'] = df_concensus_gb['answer']>=1
df_concensus_gb

,scene_id,player,question,clip,answer
0,w2l3s11,ppo,Q_eff,0,False
1,w2l3s11,ppo,Q_eff,1,False
2,w2l3s11,ppo,Q_eff,2,False
3,w2l3s11,ppo,Q_eff,3,False
4,w2l3s11,ppo,Q_eff,4,False
...,...,...,...,...,...
6299,w8l3s1,sub-03,Q_ext,45,True
6300,w8l3s1,sub-03,Q_ext,46,True
6301,w8l3s1,sub-03,Q_ext,47,True
6302,w8l3s1,sub-03,Q_ext,48,True


# Save the concensus

In [28]:
df_concensus_gb.to_csv('sourcedata/concensus_reference.csv')